<a href="https://colab.research.google.com/github/mish841/Alzheimers-Detection-Model---Grey-Hacks-2026/blob/main/AlzheimersWorkshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Before u start coding run this so u can get the DATASET folder with the correct path from github:


In [ ]:
!git clone https://github.com/mish841/Alzheimers-Detection-Model---Grey-Hacks-2026.git
%cd Alzheimers-Detection-Model---Grey-Hacks-2026

Installing Dependencies & Importing libraries:

In [ ]:
# Install dependencies (Colab usually already has these)
!pip install torchvision pillow
!pip install torchvision pillow grad-cam

#Imports
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

Load the pretrained model: We are using
[ResNet18](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html)

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Replace classifier for 2 classes
model.fc = nn.Linear(model.fc.in_features, _______)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

Image Processing:

This model excepts 224 x 224 image sizes

In [ ]:
transform = transforms.Compose([
    transforms.Resize((_______,_______)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

Use our MRI Images which were extracted from a Kaggle dataset to train the dataset a little bit

Link to dataset: [Alzheimer MRI dataset](https://www.kaggle.com/datasets/lukechugh/best-alzheimer-mri-dataset-99-accuracy?utm_source=chatgpt.com)

In [ ]:
!rm -rf dataset/train/.ipynb_checkpoints
dataset = datasets.ImageFolder("data/train", transform=transform)

train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

Freeze the CNN Layers

In [ ]:
# Freeze CNN layers
for param in model.parameters():
    param.requires_grad = _______

# Unfreeze classifier
for param in model.fc.parameters():
    param.requires_grad = ______

Training Setup

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

Train the Model

In [ ]:
train_loader = DataLoader(test_dataset, batch_size=128, shuffle=True)
epochs = 15

for epoch in range(epochs):
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.______

        outputs = _______
        loss = criterion(outputs, labels)

        loss.______
        optimizer.______

        total_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Accuracy: {accuracy:.2f}%")

Save the Model and download the model to give to the participants (they will use our TRAINED model, and just fill in the blanks of code. We will create these fill in the blanks ourselves)

In [ ]:
torch.save(model.state_dict(), "alz_model.pth")

from google.colab import files
files.download("alz_model.pth")

Load the Trained Model (Workshop Part 1)

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2)

model.load_state_dict(torch.load("alz_model.pth", map_location=device))
model = model.to(device)
model.______

In [ ]:
test_dataset = datasets.ImageFolder("data/test", transform=transform)

test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

print("Test classes:", test_dataset.classes)
print("Number of test images:", len(test_dataset))

correct = 0
total = 0

model.eval()

with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print("Test Accuracy:", accuracy, "%")

Upload MRI image

In [ ]:
from google.colab import files

uploaded = files.upload()
image_path = list(uploaded.keys())[0]

Load the image

In [ ]:
image = Image.open(image_path).convert("RGB")

plt.imshow(image)
plt.title("Uploaded MRI")
plt.axis("off")

Run the prediction

In [ ]:
input_tensor = transform(image).unsqueeze(0).to(device)

with torch.no_grad():
    output = model(input_tensor)

probs = torch.softmax(output, dim=1)
prediction = torch.argmax(probs, dim=1).item()

classes = dataset.classes

print("Prediction:", classes[prediction])
print("Confidence:", probs[0][prediction].item())

Create a probability bar chart visual

In [ ]:
labels = dataset.classes
values = probs.cpu().detach().numpy()[0]

plt.bar(labels, values)
plt.title("Model Confidence")
plt.ylabel("Probability")
plt.show()

Add a Grad-CAM Heatmap for more visualization

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

target_layer = model.layer4[-1]

cam = GradCAM(model=model, target_layers=[target_layer])

grayscale_cam = cam(______=______)[0]

rgb_img = np.array(image.resize((______,_____))) / 255.0

visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

plt.imshow(visualization)
plt.title("Grad-CAM Attention Map")
plt.axis("off")

Prediction

In [ ]:
prediction = torch.argmax(probs, dim=1).item()

classes = dataset.classes

print("Prediction:", _________)
print("Confidence:", probs[0][prediction].item())